# Dataframer: Robert Fagles's Odyssey to Pandas DF

### [—————————————pipeline—————————————]
### »——raw—»—clean—»—normalize—»—DATAFRAME——»

Here are some transformation and frequencies for future exploratory analysis of Green's Odyssey.

Columns: author, year, title, book_num, text, num_lines, num_sentences, num_words, 


In [1]:
# library imports
import os

import numpy as np
import pandas as pd

import re
import nltk

In [2]:
# Display options
pd.set_option("display.max_colwidth", None)  # Prevent truncation of long values
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.expand_frame_repr", False)  # Prevent wrapping in DataFrames

In [3]:
# Visualization libraries
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/Users/debr/English-Homer') 
import bard_visualization as viz# This will apply the visualization settings
from bard_visualization import color_palette 
sys.path.append('/Users/debr/English-Homer/functions') 
import e_nlp as e

Functions for NLP are live! use e.<function> to call them.
Download complete.
Stopwords customized:
  Added: {'n', 'seven', 'nine', "'and", 'six', 'four', 'one', 'five', 'eight', 'three', "'", 'two', 'ten'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'’', '‘', '\\', '-', '…', '“', '—', '”'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# TO UPDATE
translator = "Wilson"
book_breaker = "BOOK"

# Check Paths
filepath = f"/Users/debr/odysseys_en/Normalized_txts/Odyssey_{translator}_Normalized.txt"
output_path = f"/Users/debr/English-Homer/dataframers_by_author/{translator}_DFed/"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_path_plots = f"{output_path}/plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)

# READING FILE TO extracted_lines
with open(filepath, 'r') as file:
    extracted_lines = file.readlines()

text = "".join(extracted_lines)

In [5]:
# Function to split the text into books

#books = string_into_books(text, book_breaker)
books = e.list_into_books(extracted_lines, book_breaker)

# Verify the results
print(f"Found {len(books)} books")
for i, book in enumerate(books):
    print(f"Book {i+1} has {len(book)} lines")
    print(f"Book {i+1} starts with: {book[0]}")

Found 24 books
Book 1 has 458 lines
Book 1 starts with: Tell me about a complicated man.

Book 2 has 449 lines
Book 2 starts with: The early Dawn was born; her fingers bloomed.

Book 3 has 510 lines
Book 3 starts with: Leaving the Ocean’s streams, the Sun leapt up

Book 4 has 863 lines
Book 4 starts with: They came to Sparta, land of caves and valleys,

Book 5 has 501 lines
Book 5 starts with: Then Dawn rose up from bed with Lord Tithonus,

Book 6 has 338 lines
Book 6 starts with: Odysseus had suﬀered. In exhaustion

Book 7 has 364 lines
Book 7 starts with: Odysseus sat patiently and prayed.

Book 8 has 617 lines
Book 8 starts with: Soon Dawn appeared and touched the sky with roses.

Book 9 has 584 lines
Book 9 starts with: Wily Odysseus, the lord of lies,

Book 10 has 596 lines
Book 10 starts with: “We reached the floating island of Aeolus,

Book 11 has 660 lines
Book 11 starts with: “We reached the sea and first of all we launched

Book 12 has 472 lines
Book 12 starts with: “Our ship

In [6]:
# Books (lists) into DataFrame
df = e.book_into_df(f"{translator}", "2017", "The Odyssey", books)

# Apply functions & add new columns
df['num_lines'] = df['text'].apply(e.count_lines)
df['num_sentences'] = df['text'].apply(e.count_sentences)
df['num_words'] = df['text'].apply(e.count_words)

In [7]:
# Initialize the pipeline
nlp = e.NLPPipeline(language='english')

# Customize stopwords
nlp.customize_stopwords(
    include={'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
             "'", 'n', "'and",},
    exclude={''}
)
# Customize punctuation
nlp.customize_punctuation(
    keep={'-', ""},  # Keep hyphens and apostrophes
    remove={r'…', '—','”','’','“', '‘', '-', '\\'}  # Additional characters to remove
)

# Process with default pipeline (lowercase -> tokenize -> remove punctuation -> remove stopwords)
df = nlp.process_dataframe(df, 'text', 'tokens')
df['num_tokens'] = df['tokens'].map(len)

Stopwords customized:
  Added: {'n', 'seven', 'nine', "'and", 'four', 'six', 'one', 'eight', "'", 'three', 'five', 'two', 'ten'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'’', '‘', '\\', '-', '…', '“', '—', '”'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


In [8]:
print('Type:', type(df['text'][0]))
print('Lenght:',len(df['text'][0]))
print(df['text'][:6])

Type: <class 'list'>
Lenght: 458
0                                                                                                                                                                                                                                                                                                [Tell me about a complicated man.\n, Muse, tell me how he wandered and was lost\n, when he had wrecked the holy town of Troy,\n, and where he went, and who he met, the pain\n, he suﬀered in the storms at sea, and how\n, he worked to save his life and bring his men\n, back home. He failed to keep them safe; poor fools,\n, they ate the Sun God’s cattle, and the god\n, kept them from home. Now goddess, child of Zeus,\n, tell the old story for our modern times. \n, Find the beginning.\n, All the other Greeks\n, who had survived the brutal sack of Troy\n, sailed safely home to their own wives—except\n, this man alone. Calypso, a great goddess,\n, had trapped him in her cave;

In [9]:
print('Type:', type(df['tokens'][0]))
print('Lenght:',len(df['tokens'][0]))
print(df['tokens'][:6])

Type: <class 'list'>
Lenght: 1729
0                                                                                             [tell, complicated, man, muse, tell, wandered, lost, wrecked, holy, town, troy, went, met, pain, suﬀered, storms, sea, worked, save, life, bring, men, back, home, failed, keep, safe, poor, fools, ate, sun, god, cattle, god, kept, home, goddess, child, zeus, tell, old, story, modern, times, find, beginning, greeks, survived, brutal, sack, troy, sailed, safely, home, wives, except, man, alone, calypso, great, goddess, trapped, cave, wanted, husband, year, rolled, round, gods, decreed, go, home, ithaca, troubles, still, went, man, friendless, gods, took, pity, except, poseidon, anger, never, ended, odysseus, back, home, distant, ethiopians, live, sunset, dawn, worshipping, sea, god, feast, hundred, cattle, ...]
1                                                  [early, dawn, born, fingers, bloomed, odysseus, wellbeloved, son, jumped, put, clothes, strapped, sword

In [10]:
#Boolean check for missing values
e.check_df(df)

No missing values

df columns: Index(['author', 'year', 'title', 'book_num', 'text', 'num_lines',
       'num_sentences', 'num_words', 'tokens', 'num_tokens'],
      dtype='object') 

Shape: (24, 10)


In [11]:
df

author  year        title  book_num                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [12]:
# Create output directory if it doesn't exist
output_filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{translator}_DataFrame.csv"
os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

# save df to csv
df.to_csv(output_filepath, index=False)

print(f"Normalization complete. File saved to: {output_filepath}")

Normalization complete. File saved to: /Users/debr/odysseys_en/dataframed/Odyssey_Wilson_DataFrame.csv
